# PPY - Class 08: File I/O, Generators & Custom Exceptions
**Topics:** file reading/writing, generator pipelines, file-aware decorators, custom exceptions

> **Note:** exercises use `THE HOBBIT.txt` – place it in your working directory.

## 🧠 Theory

### File I/O basics
```python
# Always use with – auto-closes the file
with open("file.txt", "r", encoding="utf-8") as f:
    content = f.read()          # whole file as string
    lines   = f.readlines()     # list of lines
    for line in f: ...          # line-by-line (most memory-efficient)

# Write modes
open("out.txt", "w")   # overwrite
open("out.txt", "a")   # append
```

### Generator pipeline (lazy)
```python
def read_lines(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield line.strip()

def words(lines):
    for line in lines:
        for w in line.split():
            yield w
```

### Custom exceptions
```python
class NoRingException(Exception):
    def __init__(self, msg="No ring found!"):
        super().__init__(msg)

raise NoRingException()
```

## Setup – load text

In [ ]:
import os

FILE = "THE HOBBIT.txt"
if os.path.exists(FILE):
    with open(FILE, encoding="utf-8") as f:
        TEXT = f.read()
    print(f"Loaded '{FILE}': {len(TEXT):,} chars")
else:
    TEXT = (
        "In a hole in the ground there lived a hobbit.\n"
        "Not a nasty dirty wet hole, filled with the ends of worms.\n"
        "Bilbo was a very well-to-do hobbit. Bilbo Baggins was his name.\n"
        "The ring glittered in the dark. My precious, said Gollum.\n"
        "Chapter 1 begins here. Bilbo started his journey.\n"
        "Chapter 2: the ring was found again.\n"
    )
    print("Using demo text. Place THE HOBBIT.txt here for real data.")

## ✏️ Section 1.2 – File I/O Basics (I-X)

In [ ]:
from collections import Counter

lines     = TEXT.strip().splitlines()
words_lst = TEXT.split()
clean_w   = [w.strip('.,!?"-') for w in words_lst]

print(f"I.   Lines:              {len(lines)}")
print(f"II.  Words:              {len(words_lst)}")
print(f"III. 'Bilbo' (exact):    {words_lst.count('Bilbo')}")

bilbo_ci = sum(1 for w in clean_w if w.lower() == "bilbo")
print(f"IV.  'Bilbo' (any case): {bilbo_ci}")
print(f"V.   Longest word:       {max(clean_w, key=len)!r}")
print(f"VI.  Unique words:       {len(set(w.lower() for w in clean_w))}")

freq = Counter(w.lower() for w in clean_w if w)
print("VII. Top 10 words:", freq.most_common(10))

avg_len = sum(len(w) for w in words_lst) / len(words_lst)
print(f"VIII. Avg word length:   {avg_len:.2f}")

sentences  = [s.strip() for s in TEXT.replace("\n", " ").split(".")]
ring_sents = [s for s in sentences if "ring" in s.lower()]
print(f"IX.  Sentences with 'ring': {len(ring_sents)}")
for s in ring_sents[:2]: print(f"     \"{s[:80]}...\"")

chapters = [l for l in lines if l.strip().lower().startswith("chapter")]
print(f"X.   Chapters found:     {len(chapters)}")

## ✏️ Section 1.2 – Bilbo concordance (XVI)

In [ ]:
words_list2 = TEXT.split()
clean2      = [w.strip('.,!?"-') for w in words_list2]
bilbo_idx   = [i for i, w in enumerate(clean2) if w.lower() == "bilbo"]

print(f"Bilbo appears {len(bilbo_idx)} time(s). Concordance:")
for i in bilbo_idx:
    before = " ".join(words_list2[max(0, i-3):i])
    after  = " ".join(words_list2[i+1:i+4])
    print(f"  ...{before} [BILBO] {after}...")

## ✏️ Section 1.3 – Generator pipeline

In [ ]:
def read_lines(text):
    for line in text.strip().splitlines():
        yield line.strip()

def nonempty(lines):
    for l in lines:
        if l: yield l

def words_gen(lines):
    for line in lines:
        for w in line.split(): yield w

def long_words_gen(words, n=5):
    for w in words:
        if len(w.strip('.,!?"-')) > n: yield w

def sentences_gen(text):
    buf = ""
    for ch in text:
        buf += ch
        if ch == ".": yield buf.strip(); buf = ""

all_words = list(words_gen(nonempty(read_lines(TEXT))))
bilbo_cnt = sum(1 for w in all_words if w.strip(".,!?") == "Bilbo")
print(f"'Bilbo' via generator: {bilbo_cnt}")

pipeline = list(long_words_gen(words_gen(nonempty(read_lines(TEXT)))))
print("Long words sample:    ", pipeline[:6])

ring_sents = [s for s in sentences_gen(TEXT) if "ring" in s.lower()]
print(f"Ring sentences:        {len(ring_sents)}")

## ✏️ Section 1.4 – Decorators + File I/O

In [ ]:
import functools

def line_by_line(func):
    def wrapper(text):
        total = 0
        for line in text.strip().splitlines():
            total += func(line.strip())
        return total
    return wrapper

@line_by_line
def contains_bilbo(line):
    return 1 if "bilbo" in line.lower() else 0

print("Lines containing 'Bilbo':", contains_bilbo(TEXT))

def chapter_reader(func):
    def wrapper(text):
        chapters, cur = [], []
        for line in text.strip().splitlines():
            if line.strip().lower().startswith("chapter") and cur:
                chapters.append("\n".join(cur)); cur = []
            cur.append(line)
        if cur: chapters.append("\n".join(cur))
        for i, ch in enumerate(chapters, 1):
            func(i, ch)
    return wrapper

@chapter_reader
def bilbo_per_chapter(num, ch_text):
    cnt = sum(1 for w in ch_text.split() if w.strip(".,!?") == "Bilbo")
    print(f"  Chapter {num}: Bilbo x{cnt}")

print("\nBilbo per chapter:")
bilbo_per_chapter(TEXT)

## ✏️ Section 1.5 – Custom Exceptions

In [ ]:
import functools

class FileEmptyException(Exception):
    def __init__(self): super().__init__("File is empty!")

class NoRingException(Exception):
    def __init__(self, v=""): super().__init__(f"No ring found! {v}")

class NoBilboException(Exception):
    def __init__(self): super().__init__("Bilbo not found!")

class LineTooLongException(Exception):
    def __init__(self, n): super().__init__(f"Line exceeds {n} chars!")

class NoPreciousException(Exception):
    def __init__(self): super().__init__("'my precious' not found!")

class WrongFilePath(Exception):
    def __init__(self): super().__init__("Path must end with .txt!")

def check_not_empty(text):
    if not text.strip(): raise FileEmptyException()
    return "OK – file not empty"

def check_ring(text):
    if "ring" not in text.lower(): raise NoRingException()
    return "Ring found!"

def count_bilbo(text):
    cnt = sum(1 for w in text.split() if w.strip(".,!?") == "Bilbo")
    if cnt == 0: raise NoBilboException()
    return f"Bilbo appears {cnt} time(s)"

def check_lines(text, max_len=200):
    for line in text.splitlines():
        if len(line) > max_len: raise LineTooLongException(max_len)
    return "All lines OK"

def require_txt(func):
    @functools.wraps(func)
    def wrapper(path, *a, **kw):
        if not path.endswith(".txt"): raise WrongFilePath()
        return func(path, *a, **kw)
    return wrapper

@require_txt
def read_file(path): return f"Reading {path}"

for fn, arg in [(check_not_empty,TEXT),(check_ring,TEXT),(count_bilbo,TEXT),(check_lines,TEXT)]:
    try:    print(f"{fn.__name__}: {fn(arg)}")
    except Exception as e: print(f"{fn.__name__}: ERROR – {e}")

for path in ["notes.csv", "hobbit.txt"]:
    try:    print(f"require_txt({path!r}): {read_file(path)}")
    except WrongFilePath as e: print(f"require_txt({path!r}): {e}")

---
## 📋 Cheatsheet - Class 08

### File I/O
```python
with open("file.txt", "r", encoding="utf-8") as f:
    content = f.read()        # whole file
    lines   = f.readlines()   # list of lines
    for line in f: ...        # lazy – best for large files

with open("out.txt", "w", encoding="utf-8") as f:
    f.write("text\n")
    f.writelines(["a\n", "b\n"])
```

### Generator pipeline (lazy)
```python
def read_lines(path):
    with open(path, encoding="utf-8") as f:
        for line in f: yield line.strip()

def words(lines):
    for line in lines:
        for w in line.split(): yield w

for w in words(read_lines("hobbit.txt")): ...
```

### Custom exceptions
```python
class MyError(Exception):
    def __init__(self, msg="Error!"):
        super().__init__(msg)

raise MyError("oops")

try:
    raise MyError()
except MyError as e:
    print(e)
```

### File decorator pattern
```python
def line_by_line(func):
    def wrapper(file_path):
        with open(file_path, encoding="utf-8") as f:
            for line in f:
                func(line.strip())
    return wrapper
```